# 🟡 Medium: MaxPool2D Forward & Backward (NumPy)

Implement max pooling with **both** directions, returning the forward output and a `backward` closure.

### Core Idea

Max pooling downsamples by keeping only the strongest activation in each window. It has **no
parameters** — so the only thing its backward pass needs is *where each maximum came from*:

$$\text{out}[n,c,i,j] = \max_{0 \le u,v < K} x[n,\,c,\,is{+}u,\,js{+}v]$$

$$\frac{\partial \text{out}}{\partial x[n,c,h,w]} = \begin{cases} 1 & (h, w) \text{ was the argmax} \\ 0 & \text{otherwise}\end{cases}$$

Max is a **router**, not a mixer: the gradient flows through the winner untouched and every loser
gets exactly zero. (Contrast average pooling, which splits the gradient evenly across $K^2$ inputs —
and note this "winner takes all" routing is exactly what ReLU and MoE top-k gating do too.)

So the forward pass must **record the argmax**; without it the backward pass would have to re-scan
the input. Store the flat index inside each window and recover the offsets with `divmod(idx, K)`.

**Why return a closure.** `backward` captures the argmax cache in its scope — the same
`(value, vjp_fn)` pattern that JAX and PyTorch's `autograd.Function` use, and a compact way to make
the forward/backward pairing explicit:

```python
out, backward = maxpool2d(x, 2, 2)
dx = backward(dout)
```

With `stride < kernel_size` windows overlap and one pixel can win several of them, so accumulate with
`+=`. And note pooling is only *approximately* translation-invariant — shifting the input by one
pixel can flip which element wins, which is exactly why strided convolutions have largely replaced it.

### Signature
```python
def maxpool2d(x, kernel_size=2, stride=2):
    # x: (N, C, H, W)
    # returns: (out, backward) where
    #   out:      (N, C, H_out, W_out)
    #   backward: callable(dout) -> dx with the shape of x
    ...
```

### Rules
- Pure **NumPy** — no PyTorch
- The forward pass must cache the argmax; `backward` may not re-derive it from `out`
- Overlapping windows accumulate with `+=`

### Example
```
x = np.array([[[[1., 2.], [3., 9.]]]])
out, backward = maxpool2d(x, 2, 2)      # out == 9.0
backward(np.array([[[[5.]]]]))          # -> [[[[0., 0.], [0., 5.]]]]
```

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def maxpool2d(x, kernel_size=2, stride=2):
    # x: (N, C, H, W)
    # returns (out, backward) where backward(dout) -> dx
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
np.random.seed(0)
x = np.random.randn(2, 3, 8, 8)
out, backward = maxpool2d(x, kernel_size=2, stride=2)
print("out shape:", out.shape, "(expect (2, 3, 4, 4))")
print("dx  shape:", backward(np.ones_like(out)).shape, "(expect (2, 3, 8, 8))")

# Gradient routing: only the winner gets it
tiny = np.array([[[[1.0, 2.0], [3.0, 9.0]]]])
o, bwd = maxpool2d(tiny, 2, 2)
print("max      :", o.ravel())
print("routed dx:", bwd(np.array([[[[5.0]]]])).ravel(), "(expect [0 0 0 5])")

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("numpy_maxpool2d")